# VascuLens AI — Coronary CT Segmentation Starter

This notebook is an experimental image branch for binary coronary lumen segmentation from ImageCAS-X CCTA volumes and masks. It is **not** a patient-level CAD classifier, does not quantify stenosis, and does not fuse CT with the clinical table. Verify the dataset layout and label license before use.

The volume-level split occurs before slice extraction to reduce patient leakage. Report Dice/IoU and inspect overlays; pixel accuracy alone can be misleading because background dominates.

In [ ]:
!pip -q install nibabel scikit-learn
import os, glob, random, json
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; print('Device:',DEVICE)


In [ ]:
ROOT='/kaggle/input'
paths=[]
for pattern in ('**/*.nii.gz','**/*.nii'):
    paths += glob.glob(os.path.join(ROOT,pattern),recursive=True)
paths=sorted(set(paths)); print('NIfTI files:',len(paths)); print('\n'.join(paths[:30]))
# Expected convention: one file per scan containing '.img.nii.gz' and one '.coronary.nii.gz'.
pairs=[]
for p in paths:
    low=p.lower()
    if '.img.nii' not in low: continue
    mask=p[:low.index('.img.nii')]+'.coronary.nii.gz'
    if not os.path.exists(mask): mask=p[:low.index('.img.nii')]+'.coronary.nii'
    if os.path.exists(mask): pairs.append((p,mask))
print('Volume-mask pairs:',len(pairs))
if len(pairs)<3: raise FileNotFoundError('Could not identify enough ImageCAS-X pairs. Inspect dataset paths and update pairing logic to match your downloaded release.')


In [ ]:
def load_nii(path): return np.asarray(nib.load(path).get_fdata(dtype=np.float32))
vol=load_nii(pairs[0][0]); mask=load_nii(pairs[0][1])
print('Volume shape:',vol.shape,'Mask shape:',mask.shape,'Mask labels:',np.unique(mask)[:20])
z=vol.shape[2]//2
plt.figure(figsize=(10,4)); plt.subplot(1,2,1); plt.imshow(vol[:,:,z],cmap='gray'); plt.title('CCTA slice'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(mask[:,:,z]>0,cmap='gray'); plt.title('Binary coronary label'); plt.axis('off'); plt.show()


In [ ]:
IMG_SIZE=256; MAX_SLICES_PER_SCAN=30; BATCH_SIZE=8; EPOCHS=8
scan_ids=np.arange(len(pairs)); train_ids,val_ids=train_test_split(scan_ids,test_size=.2,random_state=SEED)
def slice_records(ids):
    rec=[]
    for i in ids:
        v=load_nii(pairs[i][0]); m=load_nii(pairs[i][1])
        if v.shape!=m.shape: raise ValueError(f'Shape mismatch for {pairs[i]}: {v.shape} vs {m.shape}')
        zlen=v.shape[2]; candidates=np.arange(zlen)
        if zlen>MAX_SLICES_PER_SCAN:
            candidates=np.linspace(0,zlen-1,MAX_SLICES_PER_SCAN).astype(int)
        for z in candidates: rec.append((pairs[i][0],pairs[i][1],int(z)))
    return rec
train_records=slice_records(train_ids); val_records=slice_records(val_ids)
print('Train scans:',len(train_ids),'validation scans:',len(val_ids),'slice counts:',len(train_records),len(val_records))
class CoronarySlices(Dataset):
    def __init__(self,records): self.records=records; self.cache={}
    def __len__(self): return len(self.records)
    def __getitem__(self,idx):
        vp,mp,z=self.records[idx]; key=(vp,mp)
        if key not in self.cache: self.cache[key]=(load_nii(vp),load_nii(mp))
        v,m=self.cache[key]; x=v[:,:,z]; y=(m[:,:,z]>0).astype(np.float32)
        lo,hi=np.percentile(x,[1,99]); x=np.clip(x,lo,hi); x=(x-lo)/(hi-lo+1e-6)
        x=torch.tensor(x,dtype=torch.float32)[None,None]; y=torch.tensor(y,dtype=torch.float32)[None,None]
        x=F.interpolate(x,size=(IMG_SIZE,IMG_SIZE),mode='bilinear',align_corners=False)[0]
        y=F.interpolate(y,size=(IMG_SIZE,IMG_SIZE),mode='nearest')[0]
        return x,y
train_loader=DataLoader(CoronarySlices(train_records),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
val_loader=DataLoader(CoronarySlices(val_records),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)


In [ ]:
def block(a,b): return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
class UNet(nn.Module):
    def __init__(self):
        super().__init__(); self.e1=block(1,16); self.e2=block(16,32); self.e3=block(32,64); self.pool=nn.MaxPool2d(2); self.up2=nn.ConvTranspose2d(64,32,2,2); self.d2=block(64,32); self.up1=nn.ConvTranspose2d(32,16,2,2); self.d1=block(32,16); self.out=nn.Conv2d(16,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.e3(self.pool(b)); d=self.d2(torch.cat([self.up2(c),b],1)); e=self.d1(torch.cat([self.up1(d),a],1)); return self.out(e)
model=UNet().to(DEVICE); optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3)
def loss_fn(logits,y):
    bce=F.binary_cross_entropy_with_logits(logits,y); p=torch.sigmoid(logits); dice=(2*(p*y).sum((1,2,3))+1e-6)/(p.sum((1,2,3))+y.sum((1,2,3))+1e-6); return bce+(1-dice.mean())
@torch.no_grad()
def evaluate(loader):
    model.eval(); losses=[]; dices=[]; ious=[]; sens=[]; specs=[]
    for x,y in loader:
        x=x.to(DEVICE); y=y.to(DEVICE); logits=model(x); losses.append(loss_fn(logits,y).item()); p=torch.sigmoid(logits)>=.5; t=y>=.5
        inter=(p&t).sum((1,2,3)).float(); union=(p|t).sum((1,2,3)).float(); dices.extend(((2*inter+1e-6)/(p.sum((1,2,3))+t.sum((1,2,3))+1e-6)).cpu().tolist()); ious.extend(((inter+1e-6)/(union+1e-6)).cpu().tolist())
        sens.append(((p&t).sum().float()/(t.sum().float()+1e-6)).item()); specs.append((((~p)&(~t)).sum().float()/( (~t).sum().float()+1e-6)).item())
    return {'loss':float(np.mean(losses)),'dice':float(np.mean(dices)),'iou':float(np.mean(ious)),'sensitivity':float(np.mean(sens)),'specificity':float(np.mean(specs))}
history=[]; best=-1; outdir=Path('/kaggle/working/vasculens_image_artifacts'); outdir.mkdir(parents=True,exist_ok=True)
for epoch in range(EPOCHS):
    model.train(); losses=[]
    for x,y in train_loader:
        x=x.to(DEVICE); y=y.to(DEVICE); optimizer.zero_grad(); logits=model(x); loss=loss_fn(logits,y); loss.backward(); optimizer.step(); losses.append(loss.item())
    val=evaluate(val_loader); row={'epoch':epoch+1,'train_loss':float(np.mean(losses)),**{'val_'+k:v for k,v in val.items()}}; history.append(row); print(row)
    if val['dice']>best: best=val['dice']; torch.save({'model_state_dict':model.state_dict(),'img_size':IMG_SIZE,'metrics':val},outdir/'coronary_unet.pt')
hist=pd.DataFrame(history); display(hist)


In [ ]:
checkpoint=torch.load(outdir/'coronary_unet.pt',map_location=DEVICE); model.load_state_dict(checkpoint['model_state_dict']); model.eval()
batch=next(iter(val_loader)); x,y=batch; x=x.to(DEVICE)
with torch.no_grad(): pred=(torch.sigmoid(model(x))>=.5).float().cpu()
fig,axes=plt.subplots(min(3,len(x)),3,figsize=(9,3*min(3,len(x))))
if len(x)==1: axes=np.expand_dims(axes,0)
for i in range(min(3,len(x))):
    axes[i,0].imshow(x[i,0].cpu(),cmap='gray'); axes[i,0].set_title('CCTA'); axes[i,1].imshow(y[i,0],cmap='gray'); axes[i,1].set_title('Reference mask'); axes[i,2].imshow(pred[i,0],cmap='gray'); axes[i,2].set_title('Predicted mask')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout(); plt.show()
metadata={'project':'VascuLens AI','task':'binary coronary lumen segmentation','validation_split':'scan-level 80/20 split before slice extraction','best_validation_dice':float(best),'checkpoint':'coronary_unet.pt','note':'Segmentation metrics only; this is not CAD classification or confirmed stenosis.'}
(outdir/'metadata.json').write_text(json.dumps(metadata,indent=2)); hist.to_csv(outdir/'training_history.csv',index=False)
print('Saved to',outdir); print(metadata)


## Interpretation and limitations

This is a baseline image segmentation experiment. The predicted mask is not a stenosis grade or a diagnosis. Verify the exact ImageCAS release file conventions, scan-level split, mask orientation, image spacing, and labels before reporting results. Dice and IoU describe overlap with the reference segmentation; they are not CAD accuracy. The clinical table and image dataset are not fused because patient-level linkage and common targets have not been established.